# 🌿 01 Preprocessing — Sistem LSTM Bonsai

| Item       | Detail                                      |
|------------|---------------------------------------------|
| **File**   | `notebooks/01_preprocessing.ipynb`           |
| **Tujuan** | Membersihkan data sensor, normalisasi Min-Max, bentuk sekuens waktu, labeling klasifikasi penyiraman, split train/test, simpan artefak. |
| **Input**  | `data/raw/dataset_bonsai.csv`               |
| **Output** | `artifacts/data_train.npz`, `artifacts/data_test.npz`, `artifacts/scaler_bonsai.pkl`, `artifacts/label_info.json` |
| **Urutan** | Jalankan pertama (tidak ada notebook sebelumnya) |

In [1]:
# ── VERIFIKASI LINGKUNGAN ──────────────────────────────────────────────
import sys, os

assert ".venv" in sys.executable or "venv" in sys.executable, (
    "⛔ Kernel bukan dari .venv!\n"
    "Ganti kernel ke: Python (bonsai-lstm)\n"
    f"Kernel saat ini: {sys.executable}"
)
print("✅ Kernel  :", sys.executable)
print("✅ Python  :", sys.version)
# ──────────────────────────────────────────────────────────────────────

✅ Kernel  : D:\bonsai-lstm\.venv\Scripts\python.exe
✅ Python  : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
# ── IMPORT LIBRARY & KONSTANTA GLOBAL ──────────────────────────────────
import random, os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
import joblib, json

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
print(f"[SEED] Global random seed = {SEED}")

# Verifikasi versi library
import sklearn, matplotlib
try:
    import importlib.metadata
    seaborn_ver = importlib.metadata.version("seaborn")
except:
    seaborn_ver = "unknown"
    
libs = {
    "TensorFlow"  : tf.__version__,
    "Scikit-learn": sklearn.__version__,
    "Pandas"      : pd.__version__,
    "NumPy"       : np.__version__,
    "Matplotlib"  : matplotlib.__version__,
    "Seaborn"     : seaborn_ver,
}
for lib, ver in libs.items():
    print(f"  {lib:<14}: {ver}")

# Konstanta Preprocessing
DATA_PATH      = "../data/raw/dataset_bonsai.csv"
ARTIFACTS_DIR  = "../artifacts"
TRAIN_RATIO    = 0.60
LOOK_BACK      = 24
SOIL_THRESHOLD = 60.0
FEATURES       = ["temperature_c", "humidity_air_pct", "soil_moisture_pct"]

BOUNDS = {
    "temperature_c"     : (10.0, 45.0),
    "humidity_air_pct"  : (0.0,  100.0),
    "soil_moisture_pct" : (0.0,  100.0),
}
STAGNANT_WINDOW = 12

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print("\n✅ Konfigurasi siap.")

[SEED] Global random seed = 42
  TensorFlow    : 2.21.0
  Scikit-learn  : 1.4.0
  Pandas        : 2.2.0
  NumPy         : 1.26.3
  Matplotlib    : 3.8.2
  Seaborn       : 0.13.2

✅ Konfigurasi siap.


In [3]:
# ── RULE-CLEAN-01: Parsing & Pengurutan Timestamp ──────────────────────
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
print(f"[CLEAN-01] Timestamp diparse. Rentang: {df['timestamp'].iloc[0]} → {df['timestamp'].iloc[-1]}")
print(f"[CLEAN-01] Total baris awal: {len(df)}")

[CLEAN-01] Timestamp diparse. Rentang: 2026-04-27 17:12:10 → 2026-05-06 00:14:10
[CLEAN-01] Total baris awal: 5971


In [4]:
# ── RULE-CLEAN-02: Hapus Nilai Kosong (Missing Values) ──────────────────
n_null = df.isnull().sum().sum()
df = df.dropna().reset_index(drop=True)
print(f"[CLEAN-02] Nilai null dihapus: {n_null} | Sisa baris: {len(df)}")

[CLEAN-02] Nilai null dihapus: 0 | Sisa baris: 5971


In [5]:
# ── RULE-CLEAN-03: Hapus Outlier Nilai Ekstrem ─────────────────────────
mask_valid = pd.Series([True] * len(df))
for col, (lo, hi) in BOUNDS.items():
    invalid = ~df[col].between(lo, hi, inclusive="both")
    print(f"[CLEAN-03] {col}: {invalid.sum()} outlier ditemukan")
    mask_valid &= ~invalid
df = df[mask_valid].reset_index(drop=True)
print(f"[CLEAN-03] Sisa setelah hapus outlier: {len(df)}")

[CLEAN-03] temperature_c: 0 outlier ditemukan
[CLEAN-03] humidity_air_pct: 0 outlier ditemukan
[CLEAN-03] soil_moisture_pct: 0 outlier ditemukan
[CLEAN-03] Sisa setelah hapus outlier: 5971


In [6]:
# ── RULE-CLEAN-04: Hapus Segmen Stagnan (Flat Signal) ───────────────────
stagnant_mask = pd.Series([False] * len(df))
for col in FEATURES:
    rolling_std = df[col].rolling(window=STAGNANT_WINDOW).std()
    stagnant_mask |= (rolling_std == 0)
n_stagnant = stagnant_mask.sum()
df = df[~stagnant_mask].reset_index(drop=True)
print(f"[CLEAN-04] Titik stagnan dihapus: {n_stagnant} | Sisa: {len(df)}")

[CLEAN-04] Titik stagnan dihapus: 885 | Sisa: 5086


In [7]:
# ── RULE-CLEAN-05: Hapus Duplikat Timestamp ─────────────────────────────
n_dup = df.duplicated(subset="timestamp").sum()
df = df.drop_duplicates(subset="timestamp", keep="first").reset_index(drop=True)
print(f"[CLEAN-05] Duplikat timestamp dihapus: {n_dup} | Sisa: {len(df)}")

[CLEAN-05] Duplikat timestamp dihapus: 0 | Sisa: 5086


In [8]:
# ── RULE-PRE-03: Split Data Training / Testing (Kronologis) ────────────
split_idx = int(len(df) * TRAIN_RATIO)
df_train  = df.iloc[:split_idx].reset_index(drop=True)
df_test   = df.iloc[split_idx:].reset_index(drop=True)

print(f"[SPLIT] Training : {len(df_train)} baris "
      f"({df_train['timestamp'].iloc[0]} → {df_train['timestamp'].iloc[-1]})")
print(f"[SPLIT] Testing  : {len(df_test)} baris "
      f"({df_test['timestamp'].iloc[0]} → {df_test['timestamp'].iloc[-1]})")

[SPLIT] Training : 3051 baris (2026-04-27 17:12:10 → 2026-05-03 03:14:10)
[SPLIT] Testing  : 2035 baris (2026-05-03 03:16:10 → 2026-05-06 00:14:10)


In [9]:
# ── RULE-PRE-04: Normalisasi Min-Max Scaling ────────────────────────────
scaler       = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(df_train[FEATURES])
test_scaled  = scaler.transform(df_test[FEATURES])

joblib.dump(scaler, f"{ARTIFACTS_DIR}/scaler_bonsai.pkl")
print(f"[NORM] Scaler disimpan → artifacts/scaler_bonsai.pkl")
print(f"[NORM] Min per fitur   : {dict(zip(FEATURES, scaler.data_min_))}")
print(f"[NORM] Max per fitur   : {dict(zip(FEATURES, scaler.data_max_))}")

[NORM] Scaler disimpan → artifacts/scaler_bonsai.pkl
[NORM] Min per fitur   : {'temperature_c': 24.2, 'humidity_air_pct': 35.4, 'soil_moisture_pct': 0.05}
[NORM] Max per fitur   : {'temperature_c': 35.0, 'humidity_air_pct': 99.9, 'soil_moisture_pct': 99.0}


In [10]:
# ── RULE-PRE-05: Pembentukan Sekuens Waktu ─────────────────────────────
def create_sequences(data: np.ndarray, look_back: int):
    """
    Membentuk pasangan input-output untuk LSTM.
    X[i] = data[i : i+look_back, :]     shape → (look_back, n_features)
    y[i] = data[i+look_back, 2]         indeks 2 = soil_moisture_pct
    """
    X, y = [], []
    for i in range(look_back, len(data)):
        X.append(data[i - look_back : i, :])
        y.append(data[i, 2])
    return np.array(X), np.array(y)

X_train, y_train_reg = create_sequences(train_scaled, LOOK_BACK)
X_test,  y_test_reg  = create_sequences(test_scaled,  LOOK_BACK)

print(f"[SEQ] X_train shape : {X_train.shape}  → (sampel, look_back, fitur)")
print(f"[SEQ] X_test  shape : {X_test.shape}")

[SEQ] X_train shape : (3027, 24, 3)  → (sampel, look_back, fitur)
[SEQ] X_test  shape : (2011, 24, 3)


In [11]:
# ── RULE-PRE-06: Pelabelan Klasifikasi Penyiraman (Sederhana & Agronomis) ─────
def create_labels_simple(y_reg_seq: np.ndarray, full_data: np.ndarray, scaler, threshold: float) -> np.ndarray:
    """
    Labelisasi berdasarkan kondisi tanah kering saja (sederhana dan relevan agronomi).
    Tanam perlu disiram jika kelembaban tanah < threshold.
    """
    data_subset = full_data[LOOK_BACK:]
    dummy = np.zeros((len(data_subset), 3))
    dummy[:] = data_subset
    orig = scaler.inverse_transform(dummy)
    soil = orig[:, 2]  # soil_moisture_pct
    return (soil < threshold).astype(int)

y_train_cls = create_labels_simple(y_train_reg, train_scaled, scaler, SOIL_THRESHOLD)
y_test_cls  = create_labels_simple(y_test_reg, test_scaled,  scaler, SOIL_THRESHOLD)
y_train_reg = scaler.inverse_transform(train_scaled)[LOOK_BACK:, 2]
y_test_reg  = scaler.inverse_transform(test_scaled)[LOOK_BACK:, 2]

print(f"[LABEL] TRAIN → Tidak Siram (0): {(y_train_cls==0).sum()} | Siram (1): {(y_train_cls==1).sum()}")
print(f"[LABEL] TEST  → Tidak Siram (0): {(y_test_cls==0).sum()}  | Siram (1): {(y_test_cls==1).sum()}")

rasio = max((y_train_cls==0).sum(), (y_train_cls==1).sum()) / \
        max(min((y_train_cls==0).sum(), (y_train_cls==1).sum()), 1)
print(f"[LABEL] Rasio kelas (mayor:minor) = {rasio:.2f}x {'⚠️ Imbalanced' if rasio > 4 else '✅ Balanced'}")

[LABEL] TRAIN → Tidak Siram (0): 1163 | Siram (1): 1864
[LABEL] TEST  → Tidak Siram (0): 1409  | Siram (1): 602
[LABEL] Rasio kelas (mayor:minor) = 1.60x ✅ Balanced


In [12]:
# ── RULE-PRE-07: Simpan Semua Artefak Preprocessing ────────────────────
np.savez_compressed(
    f"{ARTIFACTS_DIR}/data_train.npz",
    X_train=X_train, y_train_cls=y_train_cls, y_train_reg=y_train_reg
)
np.savez_compressed(
    f"{ARTIFACTS_DIR}/data_test.npz",
    X_test=X_test, y_test_cls=y_test_cls, y_test_reg=y_test_reg
)

label_info = {
    "soil_threshold" : SOIL_THRESHOLD,
    "look_back"      : LOOK_BACK,
    "train_ratio"    : TRAIN_RATIO,
    "features"       : FEATURES,
    "n_features"     : len(FEATURES),
    "train_label_0"  : int((y_train_cls == 0).sum()),
    "train_label_1"  : int((y_train_cls == 1).sum()),
    "test_label_0"   : int((y_test_cls  == 0).sum()),
    "test_label_1"   : int((y_test_cls  == 1).sum()),
}
with open(f"{ARTIFACTS_DIR}/label_info.json", "w") as f:
    json.dump(label_info, f, indent=2)

print("[SAVE] ✅ artifacts/data_train.npz")
print("[SAVE] ✅ artifacts/data_test.npz")
print("[SAVE] ✅ artifacts/scaler_bonsai.pkl")
print("[SAVE] ✅ artifacts/label_info.json")

[SAVE] ✅ artifacts/data_train.npz
[SAVE] ✅ artifacts/data_test.npz
[SAVE] ✅ artifacts/scaler_bonsai.pkl
[SAVE] ✅ artifacts/label_info.json


## 📊 Ringkasan Preprocessing

**Data yang dihasilkan:**
- `data_train.npz` → X_train, y_train_cls, y_train_reg
- `data_test.npz`  → X_test, y_test_cls, y_test_reg
- `scaler_bonsai.pkl` → MinMaxScaler
- `label_info.json` → Metadata & threshold

**Langkah selanjutnya:**
Jalankan `notebooks/02_training.ipynb` untuk membangun dan melatih model LSTM.